# Factor Model Estimation

This notebook estimates the minimal public two-factor model using HAC-robust OLS.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import bigframes.pandas as bpd

HAC_LAGS = 8
WINSOR_P = 0.01
MIN_OBS = 8
MAX_R2 = 0.999

PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
ANALYTICS_DATASET_ID = "<YOUR_ANALYTICS_BIGQUERY_DATASET>"

def fqn(dataset_id: str, table_name: str) -> str:
    return f"{PROJECT_ID}.{dataset_id}.{table_name}"

WEEKLY_RETURNS_TABLE = fqn(ANALYTICS_DATASET_ID, "weekly_returns_vw")
REGIME_LABELS_TABLE = fqn(ANALYTICS_DATASET_ID, "regime_labels")
OUT_GLOBAL_TABLE = fqn(ANALYTICS_DATASET_ID, "beta_alpha_estimates")
OUT_REGIME_TABLE = fqn(ANALYTICS_DATASET_ID, "beta_alpha_estimates_by_regime")

In [ ]:
def winsorize(s: pd.Series, p: float = WINSOR_P) -> pd.Series:
    s2 = s.dropna()
    if s2.empty:
        return s.astype("float64")
    low, high = s2.quantile([p, 1 - p])
    return s.clip(low, high).astype("float64")

def fit_ols_hac(df: pd.DataFrame, label):
    X = sm.add_constant(df[["market_return", "fx_return"]])
    y = df["nft_return"]
    try:
        fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    except Exception:
        return None

    r2 = float(fit.rsquared)
    if np.isnan(r2) or r2 > MAX_R2:
        return None

    params = fit.params
    bse = fit.bse
    return {
        "collection": label if isinstance(label, str) else label[0],
        "regime_class": None if isinstance(label, str) else label[1],
        "alpha_hat": float(params.iloc[0]),
        "beta_market_hat": float(params.iloc[1]),
        "beta_fx_hat": float(params.iloc[2]),
        "se_alpha": float(bse.iloc[0]),
        "se_beta_market": float(bse.iloc[1]),
        "se_beta_fx": float(bse.iloc[2]),
        "r2": r2,
        "n_obs": int(fit.nobs),
    }

In [ ]:
sql = f"""
SELECT
  r.collection,
  CAST(r.week_start AS TIMESTAMP) AS week_start,
  SAFE_CAST(r.nft_return AS FLOAT64) AS nft_return,
  SAFE_CAST(r.market_return AS FLOAT64) AS market_return,
  SAFE_CAST(r.fx_return AS FLOAT64) AS fx_return,
  COALESCE(g.regime_class, 'global') AS regime_class
FROM `{WEEKLY_RETURNS_TABLE}` AS r
LEFT JOIN `{REGIME_LABELS_TABLE}` AS g
  ON r.week_start = g.week_start
WHERE r.nft_return IS NOT NULL
  AND r.market_return IS NOT NULL
  AND r.fx_return IS NOT NULL
ORDER BY r.collection, r.week_start
"""
df = bpd.read_gbq(sql).to_pandas()
df["week_start"] = pd.to_datetime(df["week_start"]).dt.tz_localize(None)

for c in ["nft_return", "market_return", "fx_return"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = winsorize(df[c], p=WINSOR_P)

df = df.dropna().reset_index(drop=True)
df["regime_class"] = df["regime_class"].fillna("global").astype("string")

In [ ]:
rows_global = []
for coll, g in df.groupby("collection"):
    gsub = g[["nft_return", "market_return", "fx_return"]].dropna()
    if len(gsub) < MIN_OBS or gsub.std().min() == 0:
        continue
    r = fit_ols_hac(gsub, coll)
    if r:
        rows_global.append(r)

df_global = pd.DataFrame(rows_global)
bpd.DataFrame(df_global).to_gbq(OUT_GLOBAL_TABLE, if_exists="replace")
print(f"Saved global estimates -> {OUT_GLOBAL_TABLE}")
df_global.head()

In [ ]:
rows_regime = []
for (coll, regime), g in df.groupby(["collection", "regime_class"]):
    gsub = g[["nft_return", "market_return", "fx_return"]].dropna()
    if len(gsub) < MIN_OBS or gsub.std().min() == 0:
        continue
    r = fit_ols_hac(gsub, (coll, regime))
    if r:
        rows_regime.append(r)

df_regime = pd.DataFrame(rows_regime)
bpd.DataFrame(df_regime).to_gbq(OUT_REGIME_TABLE, if_exists="replace")
print(f"Saved regime-wise estimates -> {OUT_REGIME_TABLE}")
df_regime.head()